This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [3]:
import great_expectations
context = great_expectations.get_context()
import logging

In [23]:
from great_expectations.notebooks.plugins.expectations.expect_queried_custom_query_to_return_num_rows import ExpectQueriedCustomQueryToReturnNumRows

DEBUG:great_expectations.expectations.registry:Registering expectation: expect_queried_custom_query_to_return_num_rows
DEBUG:great_expectations.expectations.registry:Registering atomic.diagnostic.observed_value for expectation_type expect_queried_custom_query_to_return_num_rows.
DEBUG:great_expectations.expectations.registry:Registering atomic.diagnostic.failed for expectation_type expect_queried_custom_query_to_return_num_rows.
DEBUG:great_expectations.expectations.registry:Registering renderer.diagnostic.meta_properties for expectation_type expect_queried_custom_query_to_return_num_rows.
DEBUG:great_expectations.expectations.registry:Registering renderer.diagnostic.observed_value for expectation_type expect_queried_custom_query_to_return_num_rows.
DEBUG:great_expectations.expectations.registry:Registering renderer.diagnostic.status_icon for expectation_type expect_queried_custom_query_to_return_num_rows.
DEBUG:great_expectations.expectations.registry:Registering renderer.diagnostic.u

In [4]:
import os
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']

In [5]:
import yaml

In [6]:
from datetime import date,datetime

In [41]:
logging.basicConfig(level=logging.WARN, force = True)

In [42]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [43]:
datasource_config.get("project")

'gfw-google-827'

In [44]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

In [45]:
gx_datasource.get_asset_names()

{'fragments-3.0.0',
 'messages-2.5',
 'messages-3.0.0',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_daily-3.0.0',
 'stats_daily-2.5',
 'stats_daily-3.0.0',
 'vessel_info-2.5',
 'vessel_info-3.0.0'}

In [46]:
context.list_expectation_suite_names()

['gfw-google-827.alerts.fragments.3-0-0',
 'gfw-google-827.alerts.messages.2-5',
 'gfw-google-827.alerts.messages.3-0-0',
 'gfw-google-827.alerts.satellite_timing_offsets.2-5',
 'gfw-google-827.alerts.satellite_timing_offsets.3-0-0',
 'gfw-google-827.alerts.segment_info.2-5',
 'gfw-google-827.alerts.segment_info.3-0-0',
 'gfw-google-827.alerts.segment_vessel.2-5',
 'gfw-google-827.alerts.segment_vessel.3-0-0',
 'gfw-google-827.alerts.segs_activity.2-5',
 'gfw-google-827.alerts.segs_activity.3-0-0',
 'gfw-google-827.alerts.segs_activity_daily.2-5',
 'gfw-google-827.alerts.segs_activity_daily.3-0-0',
 'gfw-google-827.alerts.ssvids_identities.2-5',
 'gfw-google-827.alerts.ssvids_identities.3-0-0',
 'gfw-google-827.alerts.ssvids_identities_daily.2-5',
 'gfw-google-827.alerts.ssvids_identities_daily.3-0-0',
 'gfw-google-827.alerts.stats_daily.2-5',
 'gfw-google-827.alerts.stats_daily.3-0-0',
 'gfw-google-827.alerts.vessel_info.2-5',
 'gfw-google-827.alerts.vessel_info.3-0-0',
 'gfw-google-8

In [47]:
current_asset_name = 'fragment'
current_asset_constraints=[es for es in context.list_expectation_suite_names() if current_asset_name in es and 'constraints' in es]
current_asset_constraints

['gfw-google-827.constraints.fragments.3-0-0']

In [48]:
current_expectation_suite_name = current_asset_constraints[0]
print(current_expectation_suite_name)
current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
gx_splitter=gx_asset.splitter
if gx_splitter is not None:
    DATE_PARTITION_COLUMN=gx_splitter.column_name
    br_options={DATE_PARTITION_COLUMN: '2023-04-01'}
else:
    br_options={}
gx_br = gx_asset.build_batch_request(br_options)


gfw-google-827.constraints.fragments.3-0-0


In [ ]:

gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)


In [49]:

gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)



In [50]:
gx_validator.expect_column_values_to_be_unique('frag_id')

  warnings.warn(



Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "result": {
    "element_count": 360338,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "success": true,
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [21]:
gx_validator.expect_column_min_to_be_between('msg_count', min_value=0, strict_min=True)

DEBUG:great_expectations.core.expectation_configuration:evaluation_parameters have already been built on this expectation


Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_a49ea33f?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_a49ea33f
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/0d0cab4d-71c0-4ef4-8d91-70a484ca5429?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 721e356ab8578589d2126ded93e84a1e
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT min(msg_coun

{
  "result": {
    "observed_value": 1
  },
  "meta": {},
  "success": true,
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [51]:
timestamp_from_id_sql = f"""
((
    SELECT 
    TIMESTAMP(STRING_AGG(arr, '-'))
    FROM UNNEST(SPLIT(frag_id, '-')) AS arr WITH OFFSET as offset
    WHERE offset BETWEEN 1 and 3
))
"""

In [54]:
gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    SELECT *
    FROM {{active_batch}}
    WHERE {timestamp_from_id_sql} != first_msg_timestamp
"""}, value=0)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "result": {
    "observed_value": 0
  },
  "meta": {},
  "success": true,
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [55]:
gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    SELECT *
    FROM {{active_batch}}
    WHERE DATE({timestamp_from_id_sql}) != DATE(timestamp)
"""}, value=0)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "result": {
    "observed_value": 0
  },
  "meta": {},
  "success": true,
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [56]:
gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    SELECT *
    FROM {{active_batch}}
    WHERE DATE({timestamp_from_id_sql}) != DATE(last_msg_timestamp)
"""}, value=0)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "result": {
    "observed_value": 0
  },
  "meta": {},
  "success": true,
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [58]:
gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    SELECT *
    FROM {{active_batch}}
    WHERE LEFT(frag_id, STRPOS(frag_id, "-")-1) != ssvid                                                           
"""}, value=0)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "result": {
    "observed_value": 360338
  },
  "meta": {},
  "success": false,
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
gx_validator.expect_column_values_to_be_unique('frag_id')
gx_validator.expect_column_values_to_not_be_null('frag_id')


In [ ]:

gx_validator.save_expectation_suite(discard_failed_expectations=False)